In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Path resolution (identical convention to the other two notebooks) ---
_cwd = Path.cwd()
if (_cwd / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "raw" / "DHS_Districts").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise FileNotFoundError(f"Could not find data/raw/DHS_Districts from cwd {_cwd}.")

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

tfr_path = PROCESSED_DIR / "tfr_panel_full.parquet"
intensity_path = PROCESSED_DIR / "district_intensity_variables.parquet"

for p in [tfr_path, intensity_path]:
    print(p, "->", p.exists())

tfr_panel = pd.read_parquet(tfr_path)
district_intensity = pd.read_parquet(intensity_path)

print("TFR panel shape:", tfr_panel.shape)
print("District intensity shape:", district_intensity.shape)

# --- Merge on stable_district_id ---
merged = tfr_panel.merge(district_intensity, on="stable_district_id", how="left", suffixes=("", "_intensity"))
print("Merged shape (pre-restriction):", merged.shape)

n_missing_intensity = merged["delta_mobile_ownership"].isna().sum()
print(f"Rows with no matching intensity data: {n_missing_intensity} / {len(merged)}")

# --- Approximate per-district standard error for delta_mobile_ownership ---
# SE(delta) = sqrt(Var(p4) + Var(p5)), each Var(p) = p(1-p)/n -- standard
# binomial-proportion SE, combined in quadrature since the two rounds are
# independent samples
p4, n4 = merged["s930_mean"], merged["n_resp_nfhs4"]
p5, n5 = merged["v169a_mean"], merged["n_resp_nfhs5"]
merged["delta_mobile_ownership_se"] = np.sqrt((p4 * (1 - p4) / n4) + (p5 * (1 - p5) / n5))

# --- Restrict to the pre-period, rateable windows only ---
# year_approx <= 2015 (strictly pre-Jio; 2016 is ambiguous given only-annual
# granularity and Jio's Sept 2016 launch, so excluded from the clean pre-period)
RATEABLE_WINDOWS = {"primary", "sensitivity"}
pre_period = merged[
    merged["window"].isin(RATEABLE_WINDOWS) & (merged["year_approx"] <= 2015)
].copy()

print("\nPre-period (rateable, year_approx<=2015) shape:", pre_period.shape)
print("Year range:", pre_period["year_approx"].min(), "-", pre_period["year_approx"].max())
print("\nRows by round:")
print(pre_period["round"].value_counts())
print("\nDistricts covered:", pre_period["stable_district_id"].nunique(), "/ 575")
print("\nTercile distribution (delta_mobile_ownership_tercile):")
print(pre_period["delta_mobile_ownership_tercile"].value_counts())

pre_period.sort_values(["stable_district_id", "round", "year_approx"]).head(10)